# Per-Subject DSAR Erasure · 2. Erasure engine (in-place, targeted)

For every **PENDING** request, and every table in the registry, the engine:
1. builds a **dynamic WHERE** that matches the subject using **all identifier columns registered for that
   table** (most-conservative match — your decision),
2. runs an **`UPDATE … SET … WHERE …`** on the **same table** (in place — not a copy), erasing only the
   PII columns of the **matched rows**:
   - `scalar_token` / `pnr_token` → set the cell to the redaction token,
   - `json_key` → apply the compiled `regexp_replace` mask (reused from the masking repo) to redact the
     subject's `name` + URL `firstName`/`lastName`/`email` **inside** the JSON, preserving structure & non-PII.

**Every other customer's rows in the same column are untouched.** The engine prints a per-table before/after
matched-row count so you can see exactly what changed. Physical purge of the raw bytes happens in
**`03_physical_purge`** (this step leaves the deleted bytes recoverable via time-travel *until* purge).

## 0. Configuration

In [3]:
dbutils.widgets.removeAll()
dbutils.widgets.text("catalog", "dkushari_uc", "1 Catalog")
dbutils.widgets.text("schema",  "allegiant_air_dsar", "2 Schema")
dbutils.widgets.text("redaction_token", "***REDACTED***", "3 Scalar erasure token")
dbutils.widgets.text("dry_run", "false", "4 Dry run (true = count only, no UPDATE)")

CATALOG = dbutils.widgets.get("catalog")
SCHEMA  = dbutils.widgets.get("schema")
TOKEN   = dbutils.widgets.get("redaction_token")
DRY     = dbutils.widgets.get("dry_run").strip().lower() == "true"
FQ      = f"{CATALOG}.{SCHEMA}"
print("Schema:", FQ, "| token:", TOKEN, "| dry_run:", DRY)

Schema: dkushari_uc.allegiant_air_dsar | token: ***REDACTED*** | dry_run: False

## 1. Load config: pending requests + registry

In [5]:
requests = [r.asDict() for r in spark.sql(
    f"SELECT * FROM {FQ}.dsar_request WHERE status = 'PENDING'").collect()]
reg = [r.asDict() for r in spark.sql(
    f"SELECT * FROM {FQ}.pii_column_registry").collect()]

# group registry rows by table
from collections import defaultdict
by_table = defaultdict(list)
for r in reg:
    by_table[r["table_name"]].append(r)

print(len(requests), "pending requests;", len(by_table), "registered tables")

10 pending requests; 5 registered tables

## 2. The JSON mask (reused from the masking repo, targeted per row)
Same `regexp_replace` chain the blanket-masking notebook compiles for `pii=name`: redact the `name` key and
the URL `firstName`/`lastName`/`email`. Here it is applied **only to matched rows** via the UPDATE's WHERE.

In [7]:
def json_mask_expr(col):
    """Native regexp_replace chain: redact name key + URL firstName/lastName/email inside the JSON string."""
    e = f"regexp_replace({col}, '(\"name\" *: *)\"[^\"]*\"', '$1\"***REDACTED***\"')"
    e = f"regexp_replace({e}, '([?&](firstName|lastName|email)=)[^&#\"]*', '$1***REDACTED***')"
    return e

## 3. Build the per-(request, table) match predicate
For each table, use only its **identifier** columns:
- `first_name`/`last_name`/`email` → equality on that column vs the subject's value
- `full_name` (single column) → `column = 'First Last'`
- `json_key` identifier → the payload string must contain the subject's email AND first AND last name

All predicates for a table are AND-ed → "all registered identifiers must match".

In [9]:
def sqlstr(v):
    return "'" + (v or "").replace("'", "''") + "'"

def match_predicate(table, req):
    """Return a SQL boolean predicate matching this subject in this table, or None if no identifiers."""
    ids = [r for r in by_table[table] if r["is_identifier"]]
    if not ids:
        return None
    preds = []
    for r in ids:
        role = r["identifier_role"]
        col  = r["column_name"]
        if r["erase_strategy"] == "json_key":
            # identifiers are embedded in the JSON string — require email + first + last all present
            preds.append(f"{col} LIKE '%' || {sqlstr(req['email'])} || '%'")
            preds.append(f"{col} LIKE '%' || {sqlstr(req['first_name'])} || '%'")
            preds.append(f"{col} LIKE '%' || {sqlstr(req['last_name'])} || '%'")
        elif role == "email":
            preds.append(f"{col} = {sqlstr(req['email'])}")
        elif role == "first_name":
            preds.append(f"{col} = {sqlstr(req['first_name'])}")
        elif role == "last_name":
            preds.append(f"{col} = {sqlstr(req['last_name'])}")
        elif role == "full_name":
            preds.append(f"{col} = {sqlstr(req['first_name'] + ' ' + req['last_name'])}")
    return " AND ".join(preds)

# preview one predicate
if requests:
    print("example predicate (customer_profile):")
    print(" ", match_predicate("customer_profile", requests[0]))

example predicate (customer_profile):
  first_name = 'Andra' AND last_name = 'Mayfield' AND email = 'andra.mayfield@example.com'

## 4. Build the SET clause (what to erase once a row matches)
Erase **every** PII column registered for the table (identifier columns AND non-identifier PII like `pnr`).

In [11]:
def set_clause(table):
    sets = []
    for r in by_table[table]:
        col = r["column_name"]
        strat = r["erase_strategy"]
        if strat in ("scalar_token", "pnr_token"):
            sets.append(f"{col} = {sqlstr(TOKEN)}")
        elif strat == "json_key":
            sets.append(f"{col} = {json_mask_expr(col)}")
    return ", ".join(sets)

for t in by_table:
    print(t, "->", set_clause(t))

customer_profile -> first_name = '***REDACTED***', last_name = '***REDACTED***', email = '***REDACTED***'
contact_email_only -> email = '***REDACTED***'
booking -> full_name = '***REDACTED***', pnr = '***REDACTED***'
loyalty_split -> first_nm = '***REDACTED***', last_nm = '***REDACTED***'
app_hits_json -> hit_payload = regexp_replace(regexp_replace(hit_payload, '("name" *: *)"[^"]*"', '$1"***REDACTED***"'), '([?&](firstName|lastName|email)=)[^&#"]*', '$1***REDACTED***')

## 5. Run the targeted in-place UPDATEs
For each pending request × registered table: count matches, then `UPDATE` in place (unless dry-run).
We accumulate an audit of rows changed per (request, table).

In [13]:
audit = []
for req in requests:
    for table in by_table:
        pred = match_predicate(table, req)
        if not pred:
            continue
        fqt = f"{FQ}.{table}"
        n = spark.sql(f"SELECT count(*) c FROM {fqt} WHERE {pred}").collect()[0]["c"]
        if n == 0:
            audit.append((req["request_id"], req["email"], table, 0, "no_match"))
            continue
        if not DRY:
            spark.sql(f"UPDATE {fqt} SET {set_clause(table)} WHERE {pred}")
        audit.append((req["request_id"], req["email"], table, n, "dry_run" if DRY else "erased"))

import pandas as pd
adf = pd.DataFrame(audit, columns=["request_id","email","table","rows_matched","action"])
print("total rows", "that would be" if DRY else "", "erased:", adf["rows_matched"].sum())
display(spark.createDataFrame(adf))

request_id,email,table,rows_matched,action
REQ-0001,andra.mayfield@example.com,customer_profile,1,erased
REQ-0001,andra.mayfield@example.com,contact_email_only,1,erased
REQ-0001,andra.mayfield@example.com,booking,1,erased
REQ-0001,andra.mayfield@example.com,loyalty_split,1,erased
REQ-0001,andra.mayfield@example.com,app_hits_json,1,erased
REQ-0002,diego.santos@example.com,customer_profile,1,erased
REQ-0002,diego.santos@example.com,contact_email_only,1,erased
REQ-0002,diego.santos@example.com,booking,1,erased
REQ-0002,diego.santos@example.com,loyalty_split,1,erased
REQ-0002,diego.santos@example.com,app_hits_json,1,erased


## 6. Spot-check: subject erased, neighbours intact
Andra Mayfield's row shows the redaction token; a background customer in the same column is unchanged.

In [15]:
print("=== customer_profile: erased subject (Andra Mayfield) ===")
display(spark.sql(f"SELECT * FROM {FQ}.customer_profile WHERE customer_id >= 1000000 LIMIT 5"))
print("=== customer_profile: a background customer (untouched) ===")
display(spark.sql(f"SELECT * FROM {FQ}.customer_profile WHERE customer_id < 10 ORDER BY customer_id LIMIT 5"))
print("=== app_hits_json: erased subject rows (name/URL redacted, appName+revenue intact) ===")
display(spark.sql(f"SELECT * FROM {FQ}.app_hits_json WHERE hit_id >= 5000000 LIMIT 3"))

customer_id,first_name,last_name,email,home_city,lifetime_revenue
1000001,***REDACTED***,***REDACTED***,***REDACTED***,Las Vegas,4200
1000003,***REDACTED***,***REDACTED***,***REDACTED***,Las Vegas,4200
1000002,***REDACTED***,***REDACTED***,***REDACTED***,Las Vegas,4200
1000004,***REDACTED***,***REDACTED***,***REDACTED***,Las Vegas,4200
1000005,***REDACTED***,***REDACTED***,***REDACTED***,Las Vegas,4200


## 7. Note
At this point the live table values are erased, but Delta keeps the old versions (time-travel) — the raw
bytes still exist on storage. **`03_physical_purge`** removes them for CCPA "no trace".